## Model Selection Demo:

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

In [2]:
# Simulate data
np.random.seed(0)
n = 1000
p = 5
X = np.random.randn(n, p)
beta = np.array([3, 2, 0, 0, 0])  # Only two variables are nonzero
y = X @ beta + np.random.randn(n) * 0.5

# Add a constant to X for intercept
X = sm.add_constant(X)

/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/3186556678.py:7: RuntimeWarning: divide by zero encountered in matmul
  y = X @ beta + np.random.randn(n) * 0.5
/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/3186556678.py:7: RuntimeWarning: overflow encountered in matmul
  y = X @ beta + np.random.randn(n) * 0.5
/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/3186556678.py:7: RuntimeWarning: invalid value encountered in matmul
  y = X @ beta + np.random.randn(n) * 0.5


In [3]:
# Best subset selection
def best_subset_selection(X, y):
    n, p = X.shape
    models = []
    
    for k in range(1, p + 1):  # Iterate over subset sizes
        for combo in combinations(range(1, p), k):  # Generate combinations of predictors
            combo = (0,) + combo  # Include the intercept
            X_subset = X[:, combo]
            model = sm.OLS(y, X_subset).fit()
            models.append((model, combo))
    
    return models


In [4]:

# Calculate metrics
def calculate_metrics(model, X, y):
    n = len(y)
    k = model.df_model  # Number of predictors, excluding intercept

    # Mallow's CP
    

    
    # AIC
    aic = model.aic
    
    # BIC
    bic = model.bic
    
    
    # PRESS (Prediction Sum of Squares)
    hat_matrix = X @ np.linalg.inv(X.T @ X) @ X.T
    residuals = model.resid
    press = np.sum((residuals / (1 - np.diag(hat_matrix))) ** 2)
    
    # Adjusted R-squared
    r2 = model.rsquared
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    
    return aic, bic, press, adj_r2, int(k) #dont consider intercept as a predictor



In [5]:
# Run best subset selection
models = best_subset_selection(X, y)

In [6]:
# Store results in pd DataFrame
results = []
for model, combo in models:
    aic, bic, press, adj_r2, num_predictors = calculate_metrics(model, X[:, combo], y)
    results.append({
        'Predictors': combo,
        'n_Predictors': num_predictors,
        'AIC': aic,
        'BIC': bic,
        'PRESS': press,
        'Adjusted R^2': adj_r2
    })

# Convert results to pd DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='n_Predictors').reset_index(drop=True)

/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/1950885446.py:18: RuntimeWarning: divide by zero encountered in matmul
  hat_matrix = X @ np.linalg.inv(X.T @ X) @ X.T
/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/1950885446.py:18: RuntimeWarning: overflow encountered in matmul
  hat_matrix = X @ np.linalg.inv(X.T @ X) @ X.T
/var/folders/8s/b3g9yw6125n1scdnf7737s_c0000gn/T/ipykernel_13817/1950885446.py:18: RuntimeWarning: invalid value encountered in matmul
  hat_matrix = X @ np.linalg.inv(X.T @ X) @ X.T


In [7]:
# Display our results
pd.set_option('display.max_columns', None)  # Show all columns
results_df #0 represents intercept – all models include intercept

,Predictors,n_Predictors,AIC,BIC,PRESS,Adjusted R^2
0,"(0, 1)",1,4168.840872,4178.656382,3785.152258,0.707386
1,"(0, 2)",1,5038.813069,5048.628579,9034.712100,0.301575
2,"(0, 3)",1,5398.209642,5408.025152,12939.470547,-0.000469
3,"(0, 4)",1,5397.641492,5407.457002,12933.138971,0.000099
4,"(0, 5)",1,5398.175148,5407.990659,12939.770161,-0.000434
5,"(0, 4, 5)",2,5399.000246,5413.723512,12950.664816,-0.000262
6,"(0, 3, 5)",2,5399.630015,5414.353281,12957.963976,-0.000892
7,"(0, 3, 4)",2,5399.106491,5413.829757,12951.514629,-0.000368
8,"(0, 2, 5)",2,5039.863992,5054.587258,9043.968854,0.301538
9,"(0, 2, 4)",2,5039.363033,5054.086299,9040.554097,0.301888


In [ ]:
# Create summary table with Number of predictors, R2_a, Cp, and Predictors
summary_df = results_df[['n_Predictors', 'Adjusted R^2', 'Cp', 'Predictors']].copy()
summary_df.columns = ['Number of predictors', 'R2_a', 'Cp', 'Predictors in the model']

# Display the summary table
summary_df

In [ ]:
# Calculate Mallow's Cp for all models
# First, fit the full model to get MSE_full
full_model = sm.OLS(y, X).fit()
mse_full = full_model.mse_resid

# Calculate Cp for each model
cp_values = []
for model, combo in models:
    n = len(y)
    k = model.df_model  # Number of predictors (excluding intercept)
    sse = np.sum(model.resid ** 2)
    cp = sse / mse_full - n + 2 * (k + 1)  # k+1 includes intercept
    cp_values.append(cp)

# Add Cp to results_df
results_df['Cp'] = cp_values